# Capstone 2 — FastMCP Desk Co-Pilot (compressed, ~2.5h)

Spec: `README.md` in this folder. Build a FastMCP server with 2 tools (1 read + 1 write); wrap with a ReAct agent via `MultiServerMCPClient`; HITL-gate the write tool.

LangChain cheatsheet: `../../langchain_langgraph.ipynb` §§1, 4, 5, 10, 12.

**Time blocks**
| Block | Section | Goal |
|---|---|---|
| 0:00–0:30 | H1 | FastMCP skeleton — both tools as stubs |
| 0:30–1:15 | H2 | Wire the read tool (`get_portfolio_pnl`) |
| 1:15–1:45 | H3 | Wire the write tool + HITL config |
| 1:45–2:15 | H4 | Agent + 2 demos (read-only, HITL approval) |
| 2:15–2:30 | H5 | Trace links + notes |

> H1–H3 are in `mcp_server.py`. Open it alongside this notebook.

## Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

HERE = Path.cwd()
MCP_SERVER = HERE / "mcp_server.py"
PROPOSALS_DIR = HERE / "proposals"

os.environ.setdefault("LANGSMITH_PROJECT", "capstone-2-mcp")

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing"
assert MCP_SERVER.exists(), f"Where is mcp_server.py? Expected at {MCP_SERVER}"
assert ".venv" in sys.executable, f"Wrong kernel: {sys.executable}"
print("python   :", sys.executable)
print("server   :", MCP_SERVER)
print("proposals:", PROPOSALS_DIR, "->", len(list(PROPOSALS_DIR.glob('*.json'))), "existing")

---
## H1–H3 — Build the FastMCP server

Do this in `mcp_server.py`, not in a notebook cell. The file has TODO markers.

**Quick smoke test before moving on** — run from a separate terminal:
```bash
npx -y @modelcontextprotocol/inspector python mcp_server.py
```
This opens a UI where you can manually call each tool and verify the outputs. Do this BEFORE wiring the agent — debugging tool implementations through an LLM is painful.

**Sanity** for the read tool:
```
get_portfolio_pnl(date='<one of your br_marks days>')
→ {date: ..., pnl_by_trade_type: {...}, total: ..., ccy: ...}
```

**Sanity** for the write tool:
```
submit_trade_proposal(spec={instrument: '10y EUR receiver swap', side: 'buy', ...})
→ proposal_id (file appears in proposals/)
```

---
## H4 — Agent + demos (1:45–2:15)

**Hints**
- `MultiServerMCPClient` connects to your FastMCP server over `stdio` transport (cheatsheet §10).
- `await client.get_tools()` is async — use `await` in a notebook cell, or wrap in `asyncio.run`.
- `create_react_agent(llm, tools, interrupt_before=['tools'])` — the `interrupt_before=['tools']` is what enables HITL.
- HITL pattern (cheatsheet §5):
  1. `agent.invoke({...}, config={'configurable': {'thread_id': 't1'}})` — returns at the interrupt
  2. Inspect `state['messages'][-1].tool_calls` — the proposed call
  3. Approve: `agent.invoke(None, config)` resumes
  4. Reject: don't resume; start a new thread
- `interrupt_before=['tools']` triggers BEFORE *any* tool — for production you'd want this only on the write tool. For this capstone, gating all tools is fine (cleaner demo).

In [ ]:
# from langchain_mcp_adapters.client import MultiServerMCPClient
# from langchain.chat_models import init_chat_model
# from langgraph.prebuilt import create_react_agent
# from langgraph.checkpoint.memory import MemorySaver

# client = MultiServerMCPClient({
#     "trade_desk": {
#         "command": sys.executable,            # use the same .venv python
#         "args": [str(MCP_SERVER)],
#         "transport": "stdio",
#     },
# })
# tools = await client.get_tools()
# print("loaded tools:", [t.name for t in tools])

# llm = init_chat_model("openai:gpt-4o-mini", temperature=0)
# memory = MemorySaver()
# agent = create_react_agent(
#     llm,
#     tools,
#     prompt="You are a rates-desk co-pilot. Use the available tools. For trade proposals, present the spec to the user before submitting.",
#     checkpointer=memory,
#     interrupt_before=["tools"],
# )

agent = ...  # your agent
tools = ...  # for visibility

### Demo 1 — read-only PnL query

The agent should call `get_portfolio_pnl`, see the result, and answer naturally.

Note: `interrupt_before=['tools']` will pause BEFORE the read too. Inspect, then resume. (In production you'd scope HITL to writes only.)

In [ ]:
# config_d1 = {"configurable": {"thread_id": "demo-1-readonly"},
#              "tags": ["capstone-2", "demo-readonly"]}
#
# # 1) invoke until interrupt
# state = agent.invoke(
#     {"messages": [{"role": "user", "content": "What was yesterday's PnL on the rates desk?"}]},
#     config=config_d1,
# )
# print("proposed tool call:", state["messages"][-1].tool_calls)
#
# # 2) approve — resume
# state = agent.invoke(None, config=config_d1)
# print("final answer:", state["messages"][-1].content)


### Demo 2 — HITL approval flow on `submit_trade_proposal`

The interesting demo. Agent proposes a trade → interrupt → human inspects → approve → write happens.

In [ ]:
# config_d2 = {"configurable": {"thread_id": "demo-2-hitl-approve"},
#              "tags": ["capstone-2", "demo-hitl-approve"]}
#
# # 1) ask agent to propose a trade
# state = agent.invoke(
#     {"messages": [{"role": "user",
#                     "content": "Propose a 10y EUR receiver swap, 100mm notional, rationale: 'long-end rich vs 5y'"}]},
#     config=config_d2,
# )
# proposed = state["messages"][-1].tool_calls
# print("proposed call:", proposed)
# # ⚠ At this point, in real life, an analyst inspects `proposed` and decides.
#
# # 2) approve
# state = agent.invoke(None, config=config_d2)
# print("final:", state["messages"][-1].content)
# print("new file in proposals/:", sorted(p.name for p in PROPOSALS_DIR.glob('*.json')))

### (Optional) Demo 2b — HITL rejection flow

If time, demonstrate rejection: don't resume, start a new thread with feedback.

In [ ]:
# config_d2b = {"configurable": {"thread_id": "demo-2b-hitl-reject"},
#               "tags": ["capstone-2", "demo-hitl-reject"]}
# # Same prompt; on interrupt, do NOT call agent.invoke(None, ...).
# # Instead, send a corrective message in the same thread:
# state = agent.invoke(
#     {"messages": [{"role": "user",
#                     "content": "Propose a 10y EUR receiver swap, 100mm notional"}]},
#     config=config_d2b,
# )
# # Inspect; reject; send correction:
# state = agent.invoke(
#     {"messages": [{"role": "user",
#                     "content": "Cancel that. Don't submit any trades."}]},
#     config=config_d2b,
# )
# print("final:", state["messages"][-1].content)

---
## H5 — Notes (2:15–2:30)

- **Three LangSmith trace links**:
  1. …
  2. …
  3. …
- **Tool routing failures observed**: which tool docstring tweak fixed which mistake?
- **What I'd add for production**: `SqliteSaver` for cross-restart persistence, recursion limit, multi-server (add public filesystem MCP), prompt-injection test, scope HITL to write tools only via a separate node. (See `README.md` → "What got cut".)